# 06. Números finais

Universo: `censo_limpo` (limpeza do [`00b_limpar_dados.ipynb`](00b_limpar_dados.ipynb),
ainda com a lista de ouro). A aplicação já tirou essa lista, então ela não
aparece em `censo_limpo_aplicacao`.

Três fatias, uma pessoa em cada. Quem cair na lista e no pareamento conta como
`lista_ouro`. A lista é `LISTA_OURO_ARQUIVO`, a mesma do 00b — não a de
avaliação do [`03b_avaliar_lista_ouro.ipynb`](03b_avaliar_lista_ouro.ipynb).

- `lista_ouro` — CPF determinístico
- `pareamento` — linha em `splink_atribuicao.parquet` ([`04_atribuir.ipynb`](04_atribuir.ipynb))
- `sem_cpf` — o resto


In [1]:
import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    LISTA_OURO_ARQUIVO,
    SPLINK_ATRIBUICAO,
    TABELA_CENSO_LIMPA,
    cpf_norm_sql,
    get_connection,
    lista_ouro_colunas,
    print_paths,
    require_input,
    require_tables,
)

print_paths()
require_input(LISTA_OURO_ARQUIVO, label='LISTA_OURO')
require_input(SPLINK_ATRIBUICAO, label='SPLINK_ATRIBUICAO (rode o 04_atribuir antes)')
con = get_connection()
require_tables(con, [TABELA_CENSO_LIMPA], notebook_origem='00b')


OUTPUT_DIR_BASE: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output
OUTPUT_DIR: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21
recorte: uf_21
CPF_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf.parquet
CENSO_PESSOAS_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pessoas_2022.parquet
CENSO_PES_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pes_nome.parquet
CPF_CPF_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf_cpf_nome.parquet
CENSO_ESPECIE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_especie.parquet
CENSO_ENDERECO_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_endereco.parquet
CENSO_FACE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_face.parquet
CENSO_LOGR_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_

In [2]:
ouro_cols = {
    r[0] for r in con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{LISTA_OURO_ARQUIVO}') LIMIT 0"
    ).fetchall()
}
id_ouro, cpf_ouro = lista_ouro_colunas(ouro_cols)

con.execute(f'''
CREATE OR REPLACE TABLE lista_ouro AS
SELECT
    CAST({id_ouro} AS VARCHAR) AS id_censo,
    {cpf_norm_sql(cpf_ouro)} AS cpf_norm
FROM read_parquet('{LISTA_OURO_ARQUIVO}')
WHERE {id_ouro} IS NOT NULL AND {cpf_ouro} IS NOT NULL
''')

con.execute(f'''
CREATE OR REPLACE TABLE atribuicao AS
SELECT unique_id_censo, unique_id_cpf, match_probability, degrau, regra
FROM read_parquet('{SPLINK_ATRIBUICAO}')
''')

n_censo = con.execute(f'SELECT COUNT(*) FROM {TABELA_CENSO_LIMPA}').fetchone()[0]
n_ouro = con.execute('SELECT COUNT(*) FROM lista_ouro').fetchone()[0]
n_atrib = con.execute('SELECT COUNT(*) FROM atribuicao').fetchone()[0]
print(f'censo_limpo: {n_censo:,}')
print(f'lista_ouro (pares): {n_ouro:,}')
print(f'atribuicao: {n_atrib:,}')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

censo_limpo: 6,589,804
lista_ouro (pares): 19,683,974
atribuicao: 3,508,083


Quem está na lista de ouro e também no parquet conta como `lista_ouro`.
`nos_dois` tem de ser zero: essa gente saiu da aplicação antes do 04.


In [3]:
display(con.execute(f'''
SELECT
    a.origem,
    coalesce(g.n, 0) AS n,
    ROUND(100.0 * coalesce(g.n, 0) / nullif(t.n_censo, 0), 1) AS pct
FROM (VALUES
    (1, 'lista_ouro'),
    (2, 'pareamento'),
    (3, 'sem_cpf')
) a(ord, origem)
CROSS JOIN (
    SELECT COUNT(*) AS n_censo FROM {TABELA_CENSO_LIMPA}
) t
LEFT JOIN (
    SELECT
        CASE
            WHEN o.id_censo IS NOT NULL THEN 'lista_ouro'
            WHEN p.unique_id_censo IS NOT NULL THEN 'pareamento'
            ELSE 'sem_cpf'
        END AS origem,
        COUNT(*) AS n
    FROM {TABELA_CENSO_LIMPA} c
    LEFT JOIN (
        SELECT DISTINCT id_censo FROM lista_ouro
    ) o ON o.id_censo = CAST(c.person_id_censo AS VARCHAR)
    LEFT JOIN (
        SELECT DISTINCT unique_id_censo FROM atribuicao
    ) p ON p.unique_id_censo = c.unique_id
    GROUP BY 1
) g ON g.origem = a.origem
ORDER BY a.ord
''').df())

n_ambos = con.execute(f'''
SELECT COUNT(*)
FROM {TABELA_CENSO_LIMPA} c
JOIN (SELECT DISTINCT id_censo FROM lista_ouro) o
  ON o.id_censo = CAST(c.person_id_censo AS VARCHAR)
JOIN (SELECT DISTINCT unique_id_censo FROM atribuicao) p
  ON p.unique_id_censo = c.unique_id
''').fetchone()[0]
print('nos_dois:', f'{n_ambos:,}')

n_censo = con.execute(f'SELECT COUNT(*) FROM {TABELA_CENSO_LIMPA}').fetchone()[0]
n_fatias = con.execute(f'''
SELECT COUNT(*)
FROM (
    SELECT
        CASE
            WHEN o.id_censo IS NOT NULL THEN 'lista_ouro'
            WHEN p.unique_id_censo IS NOT NULL THEN 'pareamento'
            ELSE 'sem_cpf'
        END AS origem
    FROM {TABELA_CENSO_LIMPA} c
    LEFT JOIN (SELECT DISTINCT id_censo FROM lista_ouro) o
      ON o.id_censo = CAST(c.person_id_censo AS VARCHAR)
    LEFT JOIN (SELECT DISTINCT unique_id_censo FROM atribuicao) p
      ON p.unique_id_censo = c.unique_id
)
''').fetchone()[0]
assert n_fatias == n_censo
assert n_ambos == 0


,origem,n,pct
0,lista_ouro,510774,7.8
1,pareamento,3508083,53.2
2,sem_cpf,2570947,39.0


nos_dois: 0


O pareamento abaixo é o parquet do 04: um Censo, um CPF.
`degrau` é o piso da escada; `regra` é o corroborador (`score` no topo).


In [4]:
display(con.execute('''
SELECT
    degrau,
    regra,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM atribuicao
GROUP BY 1, 2
ORDER BY 1 DESC, n DESC
''').df())


,degrau,regra,n,pct
0,0.990,score,2639778,75.2
1,0.975,data_primeiro,49287,1.4
2,0.975,logradouro,31470,0.9
3,0.975,data_prefixo,24254,0.7
4,0.975,mae,16511,0.5
5,0.975,nome_cep,2923,0.1
6,0.975,tokens_cep,292,0.0
7,0.950,mae,117481,3.3
8,0.950,data_primeiro,72518,2.1
9,0.950,logradouro,58918,1.7


In [5]:
display(con.execute('''
SELECT
    n_censos,
    COUNT(*) AS n_cpf,
    n_censos * COUNT(*) AS n_censo
FROM (
    SELECT unique_id_cpf, COUNT(*) AS n_censos
    FROM atribuicao
    GROUP BY 1
)
GROUP BY 1
ORDER BY 1
''').df())

n_cpf = con.execute(
    'SELECT COUNT(DISTINCT unique_id_cpf) FROM atribuicao'
).fetchone()[0]
print(f'CPFs distintos: {n_cpf:,}')


,n_censos,n_cpf,n_censo
0,1,3375268,3375268
1,2,65338,130676
2,3,713,2139


CPFs distintos: 3,441,319


A lista de ouro inteira, e quantos desses ids estão neste Censo limpo.


In [6]:
display(con.execute(f'''
SELECT
    (SELECT COUNT(*) FROM lista_ouro) AS n_pares,
    (SELECT COUNT(DISTINCT id_censo) FROM lista_ouro) AS n_id,
    (SELECT COUNT(DISTINCT cpf_norm) FROM lista_ouro) AS n_cpf,
    (
        SELECT COUNT(DISTINCT o.id_censo)
        FROM lista_ouro o
        JOIN {TABELA_CENSO_LIMPA} c
          ON CAST(c.person_id_censo AS VARCHAR) = o.id_censo
    ) AS n_id_no_censo,
    (
        SELECT COUNT(DISTINCT o.cpf_norm)
        FROM lista_ouro o
        JOIN {TABELA_CENSO_LIMPA} c
          ON CAST(c.person_id_censo AS VARCHAR) = o.id_censo
    ) AS n_cpf_no_censo
''').df())


,n_pares,n_id,n_cpf,n_id_no_censo,n_cpf_no_censo
0,19683974,19683974,19630561,510774,509964
